<a href="https://colab.research.google.com/github/sadot04/Inteligencia_artificial/blob/main/StudentsDeep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from re import X
# 0 imports
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
#import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv("student_exam_scores.csv")





In [ ]:
df.isnull().sum()

,0
student_id,0
hours_studied,0
sleep_hours,0
attendance_percent,0
previous_scores,0
exam_score,0


In [ ]:
df

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
0,S001,8.0,8.8,72.1,45,30.2
1,S002,1.3,8.6,60.7,55,25.0
2,S003,4.0,8.2,73.7,86,35.8
3,S004,3.5,4.8,95.1,66,34.0
4,S005,9.1,6.4,89.8,71,40.3
...,...,...,...,...,...,...
195,S196,10.5,5.4,94.0,87,42.7
196,S197,7.1,6.1,85.1,92,40.4
197,S198,1.6,6.9,63.8,76,28.2
198,S199,12.0,7.3,50.5,58,42.0


In [ ]:
print("Shape brute:", df.shape)
print("Cols:", list(df.columns))

Shape brute: (200, 6)
Cols: ['student_id', 'hours_studied', 'sleep_hours', 'attendance_percent', 'previous_scores', 'exam_score']


In [ ]:
SEED= 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
X = df[[
    "hours_studied", "sleep_hours", "attendance_percent", "previous_scores"
]].copy()


In [ ]:
y = df["exam_score"].astype(float).values

In [ ]:
num_cols= ["hours_studied", "sleep_hours", "attendance_percent", "previous_scores"]

En X fueron seleccionadas las variables de hours_studied, sleep_hours, attendance_percent y previous score porque consideramos que son importantes y relevantes a la hora de predecir el exam_score que es la variable objetivo en este dataset

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
from sklearn.pipeline import Pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
    ],
    remainder = "drop"
)

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
# Ajustar transformadores en train y transformar ambos

X_train = preprocess.fit_transform(X_train_df)
X_test = preprocess.transform(X_test_df)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Input dims:", X_train.shape[1])

Input dims: 4


In [ ]:
def build_model(input_dim: int) -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(164, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1)


    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae", tf.keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

Model: "sequential_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_110 (Dense)               │ (None, 32)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_79 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_111 (Dense)               │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_80 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_112 (Dense)               │ (None, 164)            │        21,156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_81 (Dropout)            │ (None, 164)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_113 (Dense)               │ (None, 1)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,705 (100.41 KB)

 Trainable params: 25,705 (100.41 KB)

 Non-trainable params: 0 (0.00 B)

En este caso lo hicimos con 4 capas , una capa que tiene 32 neuorunas, la otra que tiene 128 neuronas, otra que tiene 164 neuronas y otra que tiene 1 sola neurona , esta decision fue debido que con estos cambios realizados nos lanzaba un mejor porcentaje en R^2 donde el mayor resultado que obtuvimos fue de 0.79 , ademas le pusimos un dropout para que el modelo no memorice los datos, ademas de que para el test fue implementado 0.2 de los datos y el 0.8 para el entrenamiento

In [ ]:
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=12,
                            restore_best_weights=True),
    callbacks.ModelCheckpoint("weather_best.keras", monitor="val_loss", mode="min",
                              save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6)
]

In [ ]:
hist = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=cbs,
    verbose=1
)

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - loss: 1155.8922 - mae: 33.4004 - rmse: 33.9960 - val_loss: 1192.8080 - val_mae: 33.7881 - val_rmse: 34.5371 - learning_rate: 0.0010
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 1115.1251 - mae: 32.7635 - rmse: 33.3904 - val_loss: 1152.8663 - val_mae: 33.1695 - val_rmse: 33.9539 - learning_rate: 0.0010
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1065.8546 - mae: 31.9887 - rmse: 32.6440 - val_loss: 1101.3757 - val_mae: 32.3482 - val_rmse: 33.1870 - learning_rate: 0.0010
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 997.4445 - mae: 30.8587 - rmse: 31.5775 - val_loss: 1032.4490 - val_mae: 31.2051 - val_rmse: 32.1317 - learning_rate: 0.0010
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 922.9811 - mae: 29.5216 - rmse: 30.3743 - val_loss: 942.5880 - val_mae: 29.6307 - val_rmse: 30.7016 - learning_rate: 0.0010
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 824.4909 - mae: 27.7211 -

In [ ]:
y_pred = model.predict(X_test).ravel()

print("\nError Cuadrático Medio (MSE):", mean_squared_error(y_test, y_pred))
print("Error Absoluto Medio (MAE):", mean_absolute_error(y_test, y_pred))
print("Coeficiente de Determinación (R²):", r2_score(y_test, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

Error Cuadrático Medio (MSE): 17.012945097327183
Error Absoluto Medio (MAE): 3.2785400295257574
Coeficiente de Determinación (R²): 0.6794250416462908


In [ ]:
def predict_one(sample: dict) -> float:

    # Convertir a DataFrame con columnas en orden esperado
    s = pd.DataFrame([sample])


    # Aplicar exactamente el mismo procesamiento
    s_proc = preprocess.transform(s[X.columns])
    s_proc = s_proc.astype(np.float32)

    # Predecir
    pred = model.predict(s_proc).item()
    return pred

In [ ]:
sample = {
    "hours_studied":7.0,
    "sleep_hours":8.0,
    "attendance_percent":75.5,
    "previous_scores":80.20,
}
pred = predict_one(sample)
print(f"Predecido {pred:.2f} ")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predecido 32.83 
